# Kho theo lô và hạn sử dụng
Dữ liệu giả lập. Nhấn Shift+Enter từng ô. Dự đoán cần lấy bao nhiêu từ hai lô còn 8 và 10 hộp khi cần 12.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'run_inventory.py').is_file())
sys.path.insert(0, str(root))
from inventory_lab.db import connect, DEFAULT_DB, LAST
from inventory_lab.sql_metrics import query
db = Path(os.environ.get('INVENTORY_LAB_DB', DEFAULT_DB))
assert db.is_file(), 'Chạy python src/prepare_data.py trong PowerShell'
def read(sql, params=None):
    with connect(db) as connection:
        return pd.read_sql_query(sql, connection, params=params)

read('SELECT * FROM lots ORDER BY lot_id LIMIT 5')

`SUM() OVER` tính tồn của các lô đứng trước. Lấy phần còn thiếu, giới hạn bởi tồn của lô đang xét. Đổi mốc sau ngày 10 để thấy lô A bị loại khỏi mô phỏng.

In [ ]:
example = """WITH stock(lot, expires, qty) AS (VALUES ('A','2026-01-10',8),('B','2026-01-20',10)),
eligible AS (SELECT *,COALESCE(SUM(qty) OVER(ORDER BY expires,lot ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),0) AS prior
FROM stock WHERE expires >= ?) SELECT lot,MAX(0,MIN(qty,12-prior)) AS taken FROM eligible ORDER BY expires,lot"""
before = read(example, ('2026-01-10',))
after = read(example, ('2026-01-11',))
assert before.taken.tolist() == [8,4] and 12 - after.taken.sum() == 2
display(before, after)

Xem tồn theo lô tại mốc. Truy vấn cộng biến động trước khi nối danh mục, mỗi dòng kết quả là một lô đã nhập.

In [ ]:
from inventory_lab.metrics import checked_tables, verify_allocation
with connect(db) as connection:
    tables = checked_tables(connection, LAST)
    allocation, shortage = verify_allocation(connection, tables['lots'], 1, 12, LAST)
display(tables['lots'].head(), allocation)
print('Còn thiếu:', shortage)

Đổi mốc báo cáo, kiểm tra lô hết hạn nào vẫn còn tồn rồi đọc ba nhận xét dưới biểu đồ.

In [ ]:
from IPython.display import Image, Markdown, display
from inventory_lab.analyze import export
out = root / 'outputs'
export(db, out, LAST)
for name in ('status','expiry','product','activity'):
    display(Image(filename=str(out / f'{name}.png')))
display(Markdown((out / 'report.md').read_text(encoding='utf-8')))